# Experiment 052: Qwen2.5 0.5B Warm GRPO with B1 (Colab)

Reuse each 006 seed's stored GSM8K-SFT model and train only a new GRPO adapter. B1 is used for accuracy reward and primary evaluation. H scores the same saved outputs as a secondary check. Format reward, data, LoRA, schedule and stopping rules remain unchanged. No SFT, pretrained base evaluation or old GRPO run is repeated. A100 40GB.

## 0. Compute units and mount Drive

Fill the two cost fields from Resources. Do not close another running session's record.

In [1]:
import json, os, time
SEEDS_TO_RUN = (0,)
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 188.03
unit_rate = 5.3
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)
open_path = os.path.join(OPEN_UNITS, f'052_{session_id}.json')
if cu_before is None or unit_rate is None:
    print('Fill cu_before and unit_rate, then rerun this cell.')
elif not os.path.exists(open_path):
    record = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
              'notebook': '052_colab_0p5b_b1_warm_grpo', 'seeds': list(SEEDS_TO_RUN),
              'colab_cu_before': cu_before, 'unit_rate': unit_rate}
    with open(open_path, 'w') as handle:
        json.dump(record, handle, indent=2)
print('Run seeds:', SEEDS_TO_RUN, '| Drive:', ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Run seeds: (0,) | Drive: /content/drive/MyDrive/inm363


## 1. GPU check: A100 with 40GB required

In [2]:
import subprocess, torch
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
gpu = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(gpu, f'| {total_gb:.1f} GiB')
assert 'A100' in gpu and 39 <= total_gb < 45, '052 requires the A100 40GB variant.'


NVIDIA A100-SXM4-40GB | 39.5 GiB


## 2. Clone or update the repo (automatic)

In [3]:
import base64
import getpass
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
repo = repo_root
subprocess.run(['git', 'diff', '--exit-code', '--', '*.py', '*.yaml'], check=True)
print('Working directory:', os.getcwd())


Working directory: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

If this cell asks for a session restart, restart the session and run the notebook again.

In [4]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'trl==1.7.1',
                'transformers==5.5.4', 'peft==0.19.1', 'accelerate==1.14.0',
                'datasets==4.8.4', 'numpy==2.0.2', 'wandb', 'pyyaml'], check=True)
# No quantization is used; the preinstalled torchao conflicts with this PEFT version.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'], check=True)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
from scripts.s52_b1_warm_grpo import PINS
if 'numpy' in sys.modules and sys.modules['numpy'].__version__ != PINS['numpy']:
    raise RuntimeError('NumPy was replaced on disk. Restart the session, then run all cells again.')
from obs import versions
actual = versions()
assert all(actual.get(lib) == version for lib, version in PINS.items()), actual
from data.mmlu import MMLU_REVISION
from data.gsm8k import GSM8K_REVISION
assert MMLU_REVISION == 'c30699e8356da336a370243923dbaf21066bb9fe'
assert GSM8K_REVISION == '740312add88f781978c0658806c59bc2815b9866'
print('Library versions and dataset revisions match.')


Library versions and dataset revisions match.


## 4. W&B key (automatic)

Reminder: switch on access for `WANDB_API_KEY` under the key icon before run. This key is only used for experiment tracking.


In [5]:
import getpass
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None
if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')
os.environ['WANDB_API_KEY'] = wkey
wkey = None
print('W&B key loaded into this session.')


W&B key loaded into this session.


## 5. Check the Drive record and free space

In [6]:
import shutil
assert os.path.exists(open_path), 'Fill the compute-unit fields and rerun section 0.'
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'Drive free: {free_gb:.0f} GiB')
assert free_gb >= 10, 'Allow at least 10 GiB for new adapters and checkpoints.'


Drive free: 178 GiB


## 6. Build the data splits

In [7]:
subprocess.run([sys.executable, 'scripts/build_data.py'], check=True)


CompletedProcess(args=['/usr/bin/python3', 'scripts/build_data.py'], returncode=0)

## 7. Historical SFT starts

The stored 006 SFT adapters, merged models and lineage records must remain on Drive. A missing parent stops the run; this notebook does not retrain SFT or rebuild a merge. The next check hashes the selected starts and may take a few minutes on Drive.


In [8]:
from config import load_config, run_tag
from file_utils import read_json
for seed in (0, 1, 2):
    tag = f'006_COLAB_FULL_B05_sft_grpo_seed{seed}'
    parent = Path(ROOT) / 'checkpoints' / tag
    required = [parent / 'sft/adapter_model.safetensors', parent / 'sft/adapter_config.json',
                parent / 'sft/adapter_base.json', parent / 'sft/config_hash.txt',
                parent / 'sft_merged/config.json', parent / 'sft_merged/tokenizer.json',
                parent / 'sft_merged/tokenizer_config.json', parent / 'grpo/adapter_base.json',
                Path(ROOT) / 'results' / tag / 'manifest.json']
    assert all(path.is_file() for path in required), f'Seed {seed}: a historical SFT source file is missing.'
    assert list((parent / 'sft_merged').glob('*.safetensors')), f'Seed {seed}: merged SFT weights are missing.'
    print(seed, 'stored SFT source:', parent / 'sft_merged')


0 stored SFT source: /content/drive/MyDrive/inm363/checkpoints/006_COLAB_FULL_B05_sft_grpo_seed0/sft_merged
1 stored SFT source: /content/drive/MyDrive/inm363/checkpoints/006_COLAB_FULL_B05_sft_grpo_seed1/sft_merged
2 stored SFT source: /content/drive/MyDrive/inm363/checkpoints/006_COLAB_FULL_B05_sft_grpo_seed2/sft_merged


## 8. Wiring check

In [9]:
assert SEEDS_TO_RUN and len(set(SEEDS_TO_RUN)) == len(SEEDS_TO_RUN)
assert all(seed in (0, 1, 2) for seed in SEEDS_TO_RUN)
for seed in SEEDS_TO_RUN:
    subprocess.run([sys.executable, 'scripts/s52_b1_warm_grpo.py', '--config',
                    f'configs/052_colab_b05_b1_warm_seed{seed}.yaml', '--task', 'preflight'], check=True)


## 9. Run the selected seeds

Completed stages are verified and skipped. A registered stop skips endpoint evaluation. Technical failure or incomplete evaluation requires inspection before retrying, do not delete a stop record or change the settings to force completion.

In [ ]:
from scripts.grpo_worker import stream_subprocess
assert cu_before is not None and unit_rate is not None, 'Fill the compute-unit fields.'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# Use a fresh process for training and each seed's endpoint evaluation.
for seed in SEEDS_TO_RUN:
    for task in ('train', 'eval'):
        print(f'Seed {seed}: {task}')
        process = stream_subprocess([sys.executable, 'scripts/s52_b1_warm_grpo.py', '--config',
                                     f'configs/052_colab_b05_b1_warm_seed{seed}.yaml', '--task', task])
        if process.returncode:
            raise RuntimeError(f'Seed {seed} {task} stopped with exit code {process.returncode}; inspect its record.')


Streaming output truncated to the last 5000 lines.
│ │ C or D)       │               │               │              │           │ │
│ │ inside        │               │               │              │           │ │
│ │ <answer></an… │               │               │              │           │ │
│ │ tags.         │               │               │              │           │ │
│ │               │               │               │              │           │ │
│ │ All of the    │               │               │              │           │ │
│ │ following are │               │               │              │           │ │
│ │ examples of   │               │               │              │           │ │
│ │ secondary     │               │               │              │           │ │
│ │ drives EXCEPT │               │               │              │           │ │
│ │ A. recreation │               │               │              │           │ │
│ │ B. sex        │               │               │       

## 10. Peek at the results

In [ ]:
from config import load_config, run_tag
for seed in (0, 1, 2):
    cfg = load_config(f'configs/052_colab_b05_b1_warm_seed{seed}.yaml')
    directory = Path(ROOT) / 'results' / run_tag(cfg)
    if (directory / 'terminal_stop.json').exists():
        print(seed, 'registered stop:', read_json(directory / 'terminal_stop.json')['outcome'])
    elif (directory / 'evaluation_complete.json').exists():
        rows = [json.loads(line) for line in (directory / 'grpo_preds.jsonl').read_text().splitlines()]
        print(seed, 'completed | B1 accuracy:', round(100 * sum(r['correct'] for r in rows) / len(rows), 2))
    else:
        print(seed, 'not completed')


0 completed | B1 accuracy: 52.39
1 completed | B1 accuracy: 36.09
2 completed | B1 accuracy: 45.7


## 11. Close the units record

In [ ]:
cu_after = 180
if cu_after is None:
    print('Fill cu_after from Resources, then rerun this cell.')
elif os.path.exists(open_path):
    with open(open_path) as handle:
        record = json.load(handle)
    record.update(ts_end=time.strftime('%Y-%m-%d %H:%M'), colab_cu_after=cu_after,
                  colab_cu_used=round(record['colab_cu_before'] - cu_after, 2))
    with open(UNITS_LOG, 'a') as handle:
        handle.write(json.dumps(record) + '\n')
    os.remove(open_path)
    print('Closed this session:', record)
else:
    print('This session has no open compute-unit record.')


Closed this session: {'session_id': '20260910_171823', 'ts_start': '2026-09-10 22:01', 'notebook': '052_colab_0p5b_b1_warm_grpo', 'seeds': [1, 2], 'colab_cu_before': 163.61, 'unit_rate': 5.3, 'ts_end': '2026-09-11 08:58', 'colab_cu_after': 164.49, 'colab_cu_used': -0.88}
